In [4]:
import os
import json
import pandas as pd

def analyze_chunk_lengths(folder_path):
    stats_list = []
    
    if not os.path.exists(folder_path):
        print(f"경로를 찾을 수 없습니다: {folder_path}")
        return None
        
    # 폴더 내 모든 json 및 jsonl 파일 탐색
    files = [f for f in os.listdir(folder_path) if f.endswith('.json') or f.endswith('.jsonl')]
    
    if not files:
        print("폴더 내에 JSON 또는 JSONL 파일이 없습니다.")
        return None

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        lengths = []
        
        try:
            # 1. JSONL 파일 처리 (한 줄씩 읽기)
            if file_name.endswith('.jsonl'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            item = json.loads(line)
                            if isinstance(item, dict) and 'page_content' in item:
                                lengths.append(len(str(item['page_content'])))
                                
            # 2. 일반 JSON 파일 처리 (통째로 읽기)
            else:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if not isinstance(data, list):
                    data = [data]
                for item in data:
                    if isinstance(item, dict) and 'page_content' in item:
                        lengths.append(len(str(item['page_content'])))
            
            # 통계치 계산
            if lengths:
                stats_list.append({
                    "file_name": file_name,
                    "min_len": min(lengths),
                    "max_len": max(lengths),
                    "avg_len": round(sum(lengths) / len(lengths), 1)
                })
                
        except Exception as e:
            print(f"파일 읽기 오류 ({file_name}): {e}")

    # 데이터프레임으로 변환 후 반환
    df = pd.DataFrame(stats_list)
    return df

# --- 실행부 ---
# 원하는 폴더 경로를 지정해서 확인해 보세요!
# target_folder = "./data/04_chunks/api"
# target_folder = "./data/04_chunks/first"
target_folder = "../data/04_chunks/final"

df_stats = analyze_chunk_lengths(target_folder)

if df_stats is not None:
    print(df_stats.to_string(index=False))

파일 읽기 오류 (kb_chunks_prec.part01.jsonl): Expecting value: line 1 column 320 (char 319)
                                                   file_name  min_len  max_len  avg_len
                     2025_주택·상가건물 임대차분쟁조정 사례집_processed.json       19      500    400.0
             전월세종합지원센터_챗봇서비스_전세사기 상담 지원_서울특별시_processed.json       59      500    373.5
                  소송등 법적절차 안내문(서울시 전월세종합지원센터)_processed.json       78      500    366.3
                     2023_주택·상가건물_임대차분쟁조정_사례집_processed.json       43      500    364.3
                    [국토교통부]_2022_주택임대차분쟁조정사례집_processed.json       17      500    348.0
                     2024_주택·상가건물 임대차분쟁조정 사례집_processed.json       11      500    374.6
                     2022_주택·상가건물_임대차분쟁조정_사례집_processed.json        5      499    321.0
                   주택임대차보호법 가이드북_20200731_개정판_processed.json       29      500    372.6
                 주택임대차전월세계약시 주요 확인사항_20260313_processed.json      170      500    454.9
2025_주택 및 상가건물 임대차 상담사례집_국토교통부_한국부

In [15]:
import os
import json
import pandas as pd

def analyze_jsonl_chunk_lengths(folder_path):
    stats_list = []
    
    if not os.path.exists(folder_path):
        print(f"경로를 찾을 수 없습니다: {folder_path}")
        return None
        
    # .jsonl 파일만 추출
    files = [f for f in os.listdir(folder_path) if f.endswith('.jsonl')]
    
    if not files:
        print("폴더 내에 JSONL 파일이 없습니다.")
        return None

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        lengths = []
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # JSONL은 한 줄에 하나의 청크 객체가 들어있으므로 줄별로 루프를 돕니다.
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                        
                    item = json.loads(line)
                    if isinstance(item, dict) and 'page_content' in item:
                        lengths.append(len(str(item['page_content'])))
            
            # 해당 파일에 계산할 청크 데이터가 존재할 경우에만 통계 추가
            if lengths:
                stats_list.append({
                    "file_name": file_name,
                    "min_len": min(lengths),
                    "max_len": max(lengths),
                    "avg_len": round(sum(lengths) / len(lengths), 1)
                })
                
        except Exception as e:
            print(f"파일 읽기 오류 ({file_name}): {e}")

    # 판다스 데이터프레임으로 만들어 예쁜 표 형태로 변환
    df = pd.DataFrame(stats_list)
    return df

# --- 실행부 ---
# api 폴더 내의 jsonl 파일 통계를 내려면 아래 경로를 그대로 사용하시면 됩니다.
target_folder = "../data/04_chunks/api"

df_stats = analyze_jsonl_chunk_lengths(target_folder)

if df_stats is not None:
    # 데이터프레임의 모든 행과 생략 없는 파일명이 다 보이도록 출력 설정
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', 1000)
    
    print(df_stats.to_string(index=False))

            file_name  min_len  max_len  avg_len
kb_chunks_eflaw.jsonl        5      500    204.3


In [8]:
# 오류가 발생한 파일 경로 지정
error_file = "../data/04_chunks/api/kb_chunks_prec.part01.jsonl"

with open(error_file, 'r', encoding='utf-8') as f:
    first_line = f.readline() # 1번째 줄만 읽기
    
    # 에러가 발생한 319~320자 주변 (앞뒤로 50자씩) 출력해보기
    start = max(0, 319 - 50)
    end = min(len(first_line), 319 + 50)
    
    print("=== 에러 발생 구간 주변 텍스트 ===")
    print(first_line[start:end])
    print(" " * (319 - start) + "^ 여기 부근에서 에러 발생")

=== 에러 발생 구간 주변 텍스트 ===
ce_id": "612861", "chunk_index": 0,"page_content":: "상대방의 의사에 반하여 그의 음성을 녹음하거나, 녹음한 음성을 방송, 배포하는 등의 
                                                  ^ 여기 부근에서 에러 발생


In [13]:
import os
import json
import pandas as pd

def locate_long_chunks_jsonl(folder_path, length_threshold=2000):
    stats_list = []
    
    if not os.path.exists(folder_path):
        print(f"경로를 찾을 수 없습니다: {folder_path}")
        return None
        
    files = [f for f in os.listdir(folder_path) if f.endswith('.jsonl')]
    
    if not files:
        print("폴더 내에 JSONL 파일이 없습니다.")
        return None

    print(f"🔍 각 파일별 {length_threshold}자 이상인 라인 번호 분석 중...")
    print("-" * 80)

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        
        long_chunk_lines = []
        total_chunks = 0
        error_count = 0
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # enumerate를 사용하여 1번째 줄부터 추적 (idx + 1)
                for idx, line in enumerate(f):
                    line = line.strip()
                    if not line:
                        continue
                    
                    total_chunks += 1
                    try:
                        item = json.loads(line)
                        if isinstance(item, dict) and 'page_content' in item:
                            content_len = len(str(item['page_content']))
                            
                            # 2000자 이상인 경우 해당 라인 번호(1부터 시작)를 저장
                            if content_len >= length_threshold:
                                long_chunk_lines.append(idx + 1)
                    except json.JSONDecodeError:
                        error_count += 1
                        continue
            
            # 리스트가 너무 길어지면 표가 깨질 수 있으므로, 
            # 발견된 라인 번호 리스트를 문자열 형태로 저장합니다.
            stats_list.append({
                "file_name": file_name,
                "total_lines": total_chunks,
                "count": len(long_chunk_lines),
                "over_2000_lines": str(long_chunk_lines),  # 라인 번호 목록 위치
                "corrupted": error_count
            })
                
        except Exception as e:
            print(f"파일 접근 오류 ({file_name}): {e}")

    df = pd.DataFrame(stats_list)
    return df

# --- 실행부 ---
target_folder = "../data/04_chunks/api"
df_results = locate_long_chunks_jsonl(target_folder, length_threshold=2000)

if df_results is not None:
    # 표가 화면에 짤리지 않고 끝까지 나오도록 설정
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', 1000)
    
    print(df_results.to_string(index=False))

🔍 각 파일별 2000자 이상인 라인 번호 분석 중...
--------------------------------------------------------------------------------
            file_name  total_lines  count over_2000_lines  corrupted
kb_chunks_eflaw.jsonl         2888      0              []          0
